# Experiment 1: Time and Frequency Domain, and the Pendulum
**Somansh Goel (500123667) | Batch A (B1)**

In [ ]:
import os
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

os.makedirs("figs", exist_ok=True)
fs, T = 500.0, 2.0
t = np.arange(0, T, 1.0 / fs)
N = t.size
rng = np.random.default_rng(42)
x = (1.00 * np.sin(2 * np.pi * 5 * t)
     + 0.50 * np.sin(2 * np.pi * 20 * t)
     + 0.25 * np.sin(2 * np.pi * 50 * t)
     + 0.20 * rng.standard_normal(N))

X = np.fft.rfft(x)
f = np.fft.rfftfreq(N, d=1.0 / fs)
amp = np.abs(X) * 2.0 / N
amp[0] /= 2.0

print(f"N={N}, df={fs/N} Hz, Nyquist={fs/2} Hz")
for sf in (5, 20, 50):
    i = int(np.argmin(np.abs(f - sf)))
    print(f"Set {sf} Hz -> Measured {f[i]:.1f} Hz, amp={amp[i]:.2f}")

In [ ]:
# Aliasing demonstration: 5 Hz sine sampled at 8 Hz
fs_a = 8.0
t_a = np.arange(0, 1.0, 1.0 / fs_a)
x_a = np.sin(2 * np.pi * 5 * t_a)
f_a = np.fft.rfftfreq(t_a.size, d=1.0 / fs_a)
X_a = np.fft.rfft(x_a)
print("Aliased peak:", f_a[np.argmax(np.abs(X_a))], "Hz")

# Pendulum numerical integration
g, L = 9.81, 1.0
T0 = 2 * np.pi * np.sqrt(L / g)
print(f"Small-angle T0 = {T0:.4f} s")

def pendulum(t_val, y, b):
    theta, omega = y
    return [omega, -b * omega - (g / L) * np.sin(theta)]

def simulate(theta0_deg, b, T_sim=60.0, n_pts=12000):
    te = np.linspace(0, T_sim, n_pts)
    sol = solve_ivp(pendulum, [0, T_sim], [np.deg2rad(theta0_deg), 0.0],
                    args=(b,), t_eval=te, rtol=1e-9, atol=1e-12)
    return sol.t, sol.y[0], sol.y[1]

for a_deg in (5, 45, 95, 135, 175):
    tt, th, _ = simulate(a_deg, 0.0)
    zc = [tt[i] + th[i]/(th[i]-th[i+1])*(tt[i+1]-tt[i])
          for i in range(len(th)-1) if th[i] > 0 and th[i+1] <= 0]
    p_zc = float(np.mean(np.diff(zc)))
    fs2 = 1.0 / (tt[1] - tt[0])
    A_sp = np.abs(np.fft.rfft(th))
    fr = np.fft.rfftfreq(th.size, 1.0 / fs2)
    p_sp = 1.0 / fr[1:][np.argmax(A_sp[1:])]
    print(f"{a_deg:>3} deg -> ZC={p_zc:.3f} s, Spec={p_sp:.3f} s, Ratio={p_zc/T0:.2f}")